In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
#reading all the words
words = open('names.txt','r').read().splitlines()
print(len(words))

32033


In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [8]:
block_size = 3
def build_dataset(words):
    X,Y = [],[]
    for w in words:
        context = [0]*block_size
        for ch in w+'.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y=torch.tensor(Y)
    print(X.shape , Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))


Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])

torch.Size([182437, 3]) torch.Size([182437])
torch.Size([22781, 3]) torch.Size([22781])
torch.Size([22928, 3]) torch.Size([22928])


In [10]:
#utility function we will use later when comparing manual gradients to pytorch gradients
def cmp(s,dt,t):
    ex = torch.all(dt = t.grad).item()
    app = torch.allclose(dt,t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximateL {str(app):5s} | maxdiff: {maxdiff}')

In [13]:
n_embd = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size,n_embd), generator = g)
#Layer 1
W1 = torch.randn((n_embd * block_size ,  n_hidden), generator = g) *(5/3)  / (n_embd *block_size)**0.5
b1 = torch.randn((n_hidden), generator = g) * 0.1
#Layer 2
W2 = torch.randn(( n_hidden, vocab_size), generator = g)* 0.1
b2 = torch.randn((vocab_size), generator = g)* 0.1

#BatchNorm Parameters
bngain =  torch.randn(1,n_hidden) * 0.1 + 1.0
bnbias =  torch.randn(1,n_hidden) * 0.1
#Note: Initilized many of these parameters in non-standard ways because sometimes initilizing with e.g. all zeros could mask an incorrect implementation of the backward pass

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) #Number of parameters in total
for p in parameters:
    p.requires_grad = True



4137


In [15]:
batch_size = 32
n = batch_size 
#construct a minibatch
ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator = g)
Xb,Yb= Xtr[ix],Ytr[ix] #batch X,Y

In [ ]:
#fordward pass, "chunkated" into smaller steps that are possible to backward one at a time
emb = C[Xb] #embed the characters into vector
embcat = emb.view(emb.shape[0],-1) #concatinate the vectors
#linear layer 1
hprebn = embcat @ W1 +b1 #hidden layer preactivation
#batchnorm layer
bnmeani = 1/n*hprebn.sum(0,keepdim = True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/ (n-1)*(bndiff2).sum(0,keepdim=True) #note bessel's correction (divide by (n-1) not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
#nonlinearity
h = torch.tanh(hpreact)#hidden layer
#linearlayer 2
logits = h@W2+b2 #output layer
#cross entropy loss (same as F.cross_entropy(logits,Yb))
logit_maxes = logits.max(1,keepdims=true).values
norm_logits = logits-logits_maxes #subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1,keepdims=True)
counts_sum_inv = counts_sum**-1 #if I use (1.0/counts_sum) instead then I can't get backprop to be exact
props = counts *counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n),Yb].mean()


#pytorch backward
for p in parameters:
    p.grad = None
for t in [logprobs,probs,counts,counts_sum,counts_sum_inv,norm_logits,logits_maxes,logits, h, hpreact,bnraw,bnvar_inv,bnvar,bndiff2,bndiff,hprebn,bnmeani,embcat,emb]:
    t.retain_grad()
loss.backward()
loss
